# Chapter 3: 外れ値対策 — RANSAC, Robust Loss, GNC

**SLAM Handbook Chapter 3** — 外れ値がSLAMに与える影響と対策手法。

## このNotebookで学ぶこと

1. **外れ値の影響** — 最小二乗法が外れ値に脆弱な理由のデモ
2. **RANSAC** — フロントエンドでの外れ値除去
3. **Robust Loss関数** — Huber, Geman-McClure, Truncated Quadratic の比較
4. **IRLS** — 反復的重み付き最小二乗法
5. **GNC** — Graduated Non-Convexity（段階的非凸化）

### 対応セクション
- Section 3.1: 外れ値の原因と影響
- Section 3.2: RANSAC, Pairwise Consistency Maximization
- Section 3.3: M-Estimation, IRLS, Black-Rangarajan Duality, GNC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 3.1 外れ値が最小二乗に与える影響

残差の二乗和 $\sum_i r_i^2$ を最小化する最小二乗法は、外れ値の大きな残差が **二乗で効く** ため、たった1つの外れ値で解が大きく歪みます。

In [ ]:
# =============================================================
# 外れ値の影響デモ: 直線フィッティング
# =============================================================
np.random.seed(42)

# 真の直線 y = 2x + 1
n_inliers = 50
n_outliers = 5
x_in = np.random.uniform(0, 10, n_inliers)
y_in = 2 * x_in + 1 + np.random.normal(0, 0.5, n_inliers)

# 外れ値（全く関係ない位置）
x_out = np.random.uniform(0, 10, n_outliers)
y_out = np.random.uniform(15, 25, n_outliers)

x_all = np.concatenate([x_in, x_out])
y_all = np.concatenate([y_in, y_out])
is_outlier = np.concatenate([np.zeros(n_inliers), np.ones(n_outliers)]).astype(bool)

# 最小二乗フィット（外れ値あり/なし）
A_in = np.column_stack([x_in, np.ones(n_inliers)])
A_all = np.column_stack([x_all, np.ones(len(x_all))])

params_clean, _, _, _ = np.linalg.lstsq(A_in, y_in, rcond=None)
params_outlier, _, _, _ = np.linalg.lstsq(A_all, y_all, rcond=None)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.scatter(x_in, y_in, c='blue', s=30, label='Inliers', alpha=0.6)
ax.scatter(x_out, y_out, c='red', s=80, marker='x', linewidths=2, label='Outliers', zorder=5)

x_line = np.linspace(0, 10, 100)
ax.plot(x_line, params_clean[0]*x_line + params_clean[1], 'g-', linewidth=2,
        label=f'LS (inliers only): y={params_clean[0]:.2f}x+{params_clean[1]:.2f}')
ax.plot(x_line, params_outlier[0]*x_line + params_outlier[1], 'r--', linewidth=2,
        label=f'LS (with outliers): y={params_outlier[0]:.2f}x+{params_outlier[1]:.2f}')
ax.plot(x_line, 2*x_line + 1, 'k:', linewidth=1, alpha=0.5, label='True: y=2x+1')

ax.legend(fontsize=10)
ax.set_title(f'最小二乗法の外れ値脆弱性: {n_outliers}/{n_inliers+n_outliers}個の外れ値で解が歪む',
             fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.2 RANSAC (Random Sample Consensus)

**SLAM Handbook Section 3.2.1**: RANSACはフロントエンドで外れ値を除去する定番手法。

### アルゴリズム
1. ランダムに最小数のサンプル（直線なら2点）を選ぶ
2. そのサンプルからモデルを推定
3. 全データ点のうちモデルに合致するもの（inlier）を数える
4. 1-3を繰り返し、最大のinlier集合を返す

必要な反復回数: $k = \frac{\log(1-p)}{\log(1-\omega^n)}$ （$p$=成功確率, $\omega$=inlier比率, $n$=最小サンプル数）

In [ ]:
# =============================================================
# RANSAC 実装
# =============================================================

def ransac_line(x, y, n_iter=100, threshold=1.0):
    """RANSAC for line fitting (y = ax + b)"""
    best_inliers = []
    best_params = None
    
    for _ in range(n_iter):
        # 1. ランダムに2点選ぶ
        idx = np.random.choice(len(x), 2, replace=False)
        x_s, y_s = x[idx], y[idx]
        
        # 2. 2点から直線を推定
        if abs(x_s[1] - x_s[0]) < 1e-10:
            continue
        a = (y_s[1] - y_s[0]) / (x_s[1] - x_s[0])
        b = y_s[0] - a * x_s[0]
        
        # 3. 全点の残差を計算し、inlierを判定
        residuals = np.abs(y - (a * x + b))
        inliers = np.where(residuals < threshold)[0]
        
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_params = (a, b)
    
    # 最終的にinlierだけで再推定
    if best_params is not None and len(best_inliers) > 1:
        A = np.column_stack([x[best_inliers], np.ones(len(best_inliers))])
        best_params, _, _, _ = np.linalg.lstsq(A, y[best_inliers], rcond=None)
    
    return best_params, best_inliers

# RANSAC実行
params_ransac, inlier_idx = ransac_line(x_all, y_all, n_iter=200, threshold=1.5)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# inlier/outlier判定の可視化
outlier_idx = np.setdiff1d(np.arange(len(x_all)), inlier_idx)
ax.scatter(x_all[inlier_idx], y_all[inlier_idx], c='blue', s=30, label=f'RANSAC inliers ({len(inlier_idx)})')
ax.scatter(x_all[outlier_idx], y_all[outlier_idx], c='red', s=80, marker='x',
           linewidths=2, label=f'RANSAC outliers ({len(outlier_idx)})')

ax.plot(x_line, params_outlier[0]*x_line + params_outlier[1], 'r--', linewidth=1.5,
        alpha=0.5, label='LS (with outliers)')
ax.plot(x_line, params_ransac[0]*x_line + params_ransac[1], 'g-', linewidth=2.5,
        label=f'RANSAC: y={params_ransac[0]:.2f}x+{params_ransac[1]:.2f}')
ax.plot(x_line, 2*x_line + 1, 'k:', linewidth=1, alpha=0.5, label='True: y=2x+1')

ax.legend(fontsize=10)
ax.set_title('RANSAC: 外れ値を自動検出して除去', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Robust Loss関数の比較

**SLAM Handbook Section 3.3, Fig. 3.5**: 二乗損失の代わりに、大きな残差を抑制する損失関数を使う。

$$\mathbf{x}^{\text{MEST}} = \arg\min_{\mathbf{x}} \sum_i \rho(r_i(\mathbf{x}))$$

ここで $\rho$ がrobust loss。影響関数 $\psi(r) = \frac{\partial \rho}{\partial r}$ が大きな$r$で小さくなることが重要。

In [ ]:
# =============================================================
# Robust Loss関数の定義と可視化 (SLAM Handbook Fig. 3.5)
# =============================================================

def quadratic_loss(r, c=None):
    return r**2

def huber_loss(r, c=1.0):
    return np.where(np.abs(r) <= c, r**2, 2*c*np.abs(r) - c**2)

def geman_mcclure_loss(r, c=1.0):
    return c**2 * r**2 / (c**2 + r**2)

def truncated_quadratic_loss(r, c=1.0):
    return np.minimum(r**2, c**2)

def tukey_biweight_loss(r, c=4.685):
    return np.where(np.abs(r) <= c,
                    c**2/6 * (1 - (1 - (r/c)**2)**3),
                    c**2/6)

# IRLS用のweight関数
def huber_weight(r, c=1.0):
    return np.where(np.abs(r) <= c, 1.0, c / np.abs(r))

def geman_mcclure_weight(r, c=1.0):
    return (c**2 / (c**2 + r**2))**2

def truncated_quadratic_weight(r, c=1.0):
    return np.where(r**2 < c**2, 1.0, 0.0)

# 可視化
r = np.linspace(-5, 5, 500)
c = 1.5  # しきい値

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

losses = [
    ('Quadratic\n$r^2$', quadratic_loss, None),
    ('Huber', huber_loss, c),
    ('Geman-McClure\n$\\frac{c^2 r^2}{c^2+r^2}$', geman_mcclure_loss, c),
    ("Tukey's Biweight", tukey_biweight_loss, 3.0),
    ('Truncated Quadratic\n$\\min(r^2, c^2)$', truncated_quadratic_loss, c),
]

for idx, (name, func, param) in enumerate(losses):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    if param is not None:
        vals = func(r, param)
    else:
        vals = func(r)
    ax.plot(r, vals, 'b-', linewidth=2)
    ax.plot(r, r**2, 'k--', alpha=0.3, linewidth=1)  # quadratic for reference
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('r')
    ax.set_ylabel('ρ(r)')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.5, 15)

# 最後のパネル: 全lossの比較
ax = axes[1, 2]
ax.plot(r, r**2, 'k-', linewidth=2, label='Quadratic', alpha=0.5)
ax.plot(r, huber_loss(r, c), 'b-', linewidth=2, label='Huber')
ax.plot(r, geman_mcclure_loss(r, c), 'r-', linewidth=2, label='Geman-McClure')
ax.plot(r, truncated_quadratic_loss(r, c), 'g-', linewidth=2, label='Truncated Quad.')
ax.set_title('All Losses Compared', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(-0.5, 8)
ax.grid(True, alpha=0.3)

plt.suptitle('Robust Loss Functions (SLAM Handbook Fig. 3.5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.4 IRLS (Iteratively Reweighted Least Squares)

**SLAM Handbook Section 3.3.1, Eq. 3.11-3.12**: Robust Lossを直接最適化する代わりに、**重み付き最小二乗** を反復的に解く。

各反復で重み $w_i$ を更新:

$$w_i(\mathbf{x}^{(t)}) = \frac{\psi(r_i(\mathbf{x}^{(t)}))}{2 r_i(\mathbf{x}^{(t)})}$$

そして重み付き最小二乗を解く:

$$\mathbf{x}^{(t+1)} = \arg\min_{\mathbf{x}} \sum_i w_i^{(t)} r_i^2(\mathbf{x})$$

In [ ]:
# =============================================================
# IRLS for robust line fitting
# =============================================================

def irls_line_fit(x, y, weight_func, c=1.5, n_iter=30):
    """IRLS for line fitting with robust loss"""
    # 初期推定: 通常の最小二乗
    A = np.column_stack([x, np.ones(len(x))])
    params, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
    
    history = []
    for iteration in range(n_iter):
        # 残差
        residuals = y - A @ params
        
        # 重みの更新
        w = weight_func(residuals, c)
        w = np.clip(w, 1e-10, None)  # ゼロ除算防止
        
        # 重み付き最小二乗
        W = np.diag(w)
        params = np.linalg.solve(A.T @ W @ A, A.T @ W @ y)
        
        cost = np.sum(w * residuals**2)
        history.append(cost)
    
    return params, w, history

# 3つのロバスト損失で比較
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, w_func) in zip(axes, [
    ('Huber', huber_weight),
    ('Geman-McClure', geman_mcclure_weight),
    ('Truncated Quadratic', truncated_quadratic_weight),
]):
    params_irls, weights, hist = irls_line_fit(x_all, y_all, w_func, c=1.5)
    
    # 重みで色分け
    scatter = ax.scatter(x_all, y_all, c=weights, cmap='RdYlGn', s=50,
                         edgecolors='black', linewidths=0.5, vmin=0, vmax=1)
    plt.colorbar(scatter, ax=ax, label='Weight $w_i$', shrink=0.8)
    
    ax.plot(x_line, params_irls[0]*x_line + params_irls[1], 'b-', linewidth=2.5,
            label=f'IRLS: y={params_irls[0]:.2f}x+{params_irls[1]:.2f}')
    ax.plot(x_line, 2*x_line + 1, 'k:', alpha=0.5, label='True')
    
    ax.legend(fontsize=9)
    ax.set_title(f'IRLS + {name}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("→ 緑=高い重み(inlier), 赤=低い重み(outlier)")
print("→ Geman-McClureとTruncated Quadraticは外れ値の重みをほぼ0にする")

## 3.5 GNC (Graduated Non-Convexity)

**SLAM Handbook Section 3.3.4**: IRLSの初期値依存性を改善する手法。

ロバスト損失関数 $\rho$ をパラメータ $\mu$ で平滑化した $\rho_\mu$ を使う:
- $\mu$ が小さい → $\rho_\mu$ は凸（最小二乗に近い）→ 良い初期値が不要
- $\mu$ が大きい → $\rho_\mu$ は元の非凸ロバスト損失に近づく → 外れ値を拒絶

**GNC Geman-McClure** (SLAM Handbook Eq. 3.25):

$$\rho_\mu(r) = \frac{\mu \beta^2 r^2}{\mu \beta^2 + r^2}$$

$\mu \to \infty$ で凸、$\mu = 1$ で元のGM損失。各反復で $\mu$ を $\mu / \gamma$ に減少させる。

In [ ]:
# =============================================================
# GNC Geman-McClure の可視化 (SLAM Handbook Fig. 3.8b)
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GNC GM損失の形状変化
ax = axes[0]
r = np.linspace(-5, 5, 500)
beta = 1.5
for mu in [100, 10, 3, 1.5, 1.0]:
    rho_mu = mu * beta**2 * r**2 / (mu * beta**2 + r**2)
    ax.plot(r, rho_mu, linewidth=2, label=f'μ={mu}')

ax.set_title('GNC Geman-McClure Loss\n(μ↓ で非凸化)', fontweight='bold')
ax.set_xlabel('r'); ax.set_ylabel('ρ_μ(r)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(-0.2, 5)

# GNC weight update (Eq. 3.20 with mu)
ax = axes[1]
r_pos = np.linspace(0.01, 5, 200)
for mu in [100, 10, 3, 1.5, 1.0]:
    w = (mu * beta**2 / (mu * beta**2 + r_pos**2))**2
    ax.plot(r_pos, w, linewidth=2, label=f'μ={mu}')

ax.set_title('GNC Weight Function\n(μ↓ で外れ値の重みが急減)', fontweight='bold')
ax.set_xlabel('|r|'); ax.set_ylabel('w(r)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# GNC 実装: line fitting
# =============================================================

def gnc_gm_line_fit(x, y, beta=1.5, mu_init=100, gamma=1.4, n_outer=20, n_inner=5):
    """GNC with Geman-McClure loss for line fitting"""
    A = np.column_stack([x, np.ones(len(x))])
    params, _, _, _ = np.linalg.lstsq(A, y, rcond=None)  # 初期推定
    
    mu = mu_init
    history = []
    
    for outer in range(n_outer):
        # Inner loop: IRLS with current mu
        for inner in range(n_inner):
            residuals = y - A @ params
            # GNC GM weight
            w = (mu * beta**2 / (mu * beta**2 + residuals**2))**2
            w = np.clip(w, 1e-10, None)
            W = np.diag(w)
            params = np.linalg.solve(A.T @ W @ A, A.T @ W @ y)
        
        residuals = y - A @ params
        cost = np.sum(mu * beta**2 * residuals**2 / (mu * beta**2 + residuals**2))
        history.append((mu, cost, params.copy(), w.copy()))
        
        # μを減少させて非凸化
        mu = max(mu / gamma, 1.0)
        if mu <= 1.0:
            break
    
    return params, w, history

params_gnc, weights_gnc, gnc_history = gnc_gm_line_fit(x_all, y_all)

# GNCの各ステップを可視化
n_steps = min(len(gnc_history), 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx in range(n_steps):
    ax = axes[idx // 3, idx % 3]
    mu, cost, params_step, w_step = gnc_history[idx]
    
    scatter = ax.scatter(x_all, y_all, c=w_step, cmap='RdYlGn', s=50,
                         edgecolors='black', linewidths=0.5, vmin=0, vmax=1)
    ax.plot(x_line, params_step[0]*x_line + params_step[1], 'b-', linewidth=2)
    ax.plot(x_line, 2*x_line + 1, 'k:', alpha=0.3)
    ax.set_title(f'Step {idx+1}: μ={mu:.1f}\ny={params_step[0]:.2f}x+{params_step[1]:.2f}',
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-5, 30)

plt.suptitle('GNC (Graduated Non-Convexity): μを徐々に減少\n'
             '凸(μ大)→非凸(μ=1)で段階的に外れ値を拒絶',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"最終結果: y = {params_gnc[0]:.3f}x + {params_gnc[1]:.3f}")
print(f"真値:     y = 2.000x + 1.000")

## 3.6 演習問題

### 演習1: 外れ値率の影響
外れ値の割合を 5%, 15%, 30%, 50% に変えて、RANSAC / IRLS / GNC の性能を比較してください。どこまで耐えられますか？

### 演習2: Pose Graph SLAMへの適用
Ch02の `02_lie_group_optimization.ipynb` のPose Graphに偽のループクロージャ（外れ値エッジ）を追加し、通常のGN法が破綻する様子を確認してください。次にGNCを組み込んで外れ値を自動拒絶してみましょう。

### 演習3: Pairwise Consistency Maximization
SLAM Handbook Section 3.2.2のPCMを実装してみましょう。ループクロージャのペアが互いに整合するかをチェックし、consistency graphの最大クリークを求めます。

## まとめ

| 手法 | 使う場所 | 特徴 |
|---|---|---|
| **RANSAC** | フロントエンド | 高速、少数外れ値に有効、大きなminimal setには不向き |
| **PCM** | フロントエンド | 高外れ値率に対応、状態に依存しない |
| **Huber Loss** | バックエンド | 凸、BA等で広く使用、外れ値の影響を完全には除去しない |
| **GM/TQ Loss** | バックエンド | 非凸、外れ値を完全に拒絶可能だが局所解のリスク |
| **IRLS** | バックエンド | Robust Lossを重み付きLSで効率的に解く |
| **GNC** | バックエンド | IRLSの初期値依存性を改善、80-90%の外れ値に耐性 |

### 次のNotebook
→ Ch04: 微分可能最適化（自動微分とSLAMの接点）